# 05 - PySpark EDA, SQL Joins & Visualizations

Fills the remaining Technical Requirement gaps from the brief:
- PySpark `.describe()`, `.groupBy().agg()` for large-scale exploration
- Statistical measures (mean, median, std, skewness, kurtosis) via PySpark SQL functions
- Data profiling (null counts, cardinality, outlier detection)
- Joins performed in PySpark SQL (not Pandas) -- with a broadcast join demonstrated
- Visualizations (matplotlib/seaborn) -- converted to Pandas only at the final plotting step

**Run this AFTER `02_PySpark_Load_Clean_Partition.ipynb`** -- it reads the same flattened CSVs.


## 1. Start Spark session and load data

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("EDA_Visualizations") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

tt = spark.read.csv(r"D:\Dataset\timetables_flat.csv", header=True, inferSchema=True)
fr = spark.read.csv(r"D:\Dataset\fares_flat.csv", header=True, inferSchema=True)
di = spark.read.csv(r"D:\Dataset\catalogue\disruptions_data_catalogue.csv", header=True, inferSchema=True)
di = di.filter(F.col("Organisation") == "West of England")

print("Timetables rows:", tt.count())
print("Fares rows:", fr.count())
print("Disruptions rows:", di.count())

## 2. PySpark `.describe()` -- large-scale statistical summary

In [ ]:
tt.describe().show()

## 3. Data profiling -- null counts per column

In [ ]:
null_counts = tt.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in tt.columns])
null_counts.show()

## 4. Data profiling -- cardinality (distinct value counts)

In [ ]:
tt.select(
    F.countDistinct("line_name").alias("distinct_routes"),
    F.countDistinct("operator_noc").alias("distinct_operators"),
    F.countDistinct("stop_point_ref").alias("distinct_stops"),
    F.countDistinct("service_code").alias("distinct_service_codes")
).show()

## 5. Statistical measures via PySpark SQL functions

Mean, median, standard deviation, skewness, and kurtosis of scheduled stop times -- required by the brief's Exploratory Data Analysis section.

In [ ]:
tt.select(
    F.mean("scheduled_time_sec").alias("mean_time"),
    F.expr("percentile_approx(scheduled_time_sec, 0.5)").alias("median_time"),
    F.stddev("scheduled_time_sec").alias("stddev_time"),
    F.skewness("scheduled_time_sec").alias("skewness_time"),
    F.kurtosis("scheduled_time_sec").alias("kurtosis_time")
).show()

## 6. Outlier detection

Using the IQR method on `scheduled_time_sec` to flag unusually early/late scheduled stops.

In [ ]:
quantiles = tt.approxQuantile("scheduled_time_sec", [0.25, 0.75], 0.01)
q1, q3 = quantiles[0], quantiles[1]
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Outlier bounds: [{lower_bound}, {upper_bound}]")

outliers = tt.filter((F.col("scheduled_time_sec") < lower_bound) | (F.col("scheduled_time_sec") > upper_bound))
print("Outlier count:", outliers.count(), "out of", tt.count())

## 7. GroupBy + agg -- trips and stops per route (large-scale exploration)

In [ ]:
route_summary = tt.groupBy("line_name").agg(
    F.countDistinct("vehicle_journey_code").alias("trip_count"),
    F.countDistinct("stop_point_ref").alias("stop_count")
).orderBy(F.desc("trip_count"))

route_summary.show(25, truncate=False)

## 8. Join datasets using PySpark SQL (not Pandas)

This satisfies the brief's requirement that dataset integration use PySpark or SQL, not just Pandas.
A **broadcast join** is used since the Fares and Disruptions aggregates are small relative to Timetables -- Spark's query plan (shown below) confirms `BroadcastHashJoin` is applied.

In [ ]:
# Aggregate Fares and Disruptions in PySpark (not Pandas)
route_fares = fr.groupBy("route").agg(
    F.avg("price_gbp").alias("avg_fare"),
    F.stddev("price_gbp").alias("fare_std"),
    F.count("price_gbp").alias("fare_product_count")
).fillna(0, subset=["fare_std"])

route_disruptions = di.groupBy("Services affected").agg(F.count("*").alias("disruption_count")) \
    .withColumnRenamed("Services affected", "route_float")
route_disruptions = route_disruptions.withColumn("route", F.col("route_float").cast("int").cast("string")).drop("route_float")

# Broadcast join: route_fares and route_disruptions are small (<200 rows), route_summary side is the "large" side
joined_spark = (route_summary.withColumnRenamed("line_name", "route")
    .join(F.broadcast(route_fares), on="route", how="inner")
    .join(F.broadcast(route_disruptions), on="route", how="left")
    .fillna(0, subset=["disruption_count"])
    .withColumn("reliability_score", F.col("disruption_count") / F.col("trip_count")))

joined_spark.explain()  # confirms BroadcastHashJoin in the physical plan
joined_spark.orderBy(F.desc("disruption_count")).show(25, truncate=False)

## 9. Register as SQL views and run an equivalent join via `spark.sql()`

Demonstrates PySpark SQL query capability directly, as required by the brief's "PySpark SQL for complex queries and aggregations" point.

In [ ]:
route_summary.withColumnRenamed("line_name", "route").createOrReplaceTempView("route_summary")
route_fares.createOrReplaceTempView("route_fares")
route_disruptions.createOrReplaceTempView("route_disruptions")

sql_result = spark.sql('''
    SELECT rs.route, rs.trip_count, rs.stop_count, rf.avg_fare,
           COALESCE(rd.disruption_count, 0) AS disruption_count
    FROM route_summary rs
    JOIN route_fares rf ON rs.route = rf.route
    LEFT JOIN route_disruptions rd ON rs.route = rd.route
    ORDER BY rs.trip_count DESC
''')
sql_result.show(25, truncate=False)

## 10. Convert to Pandas ONLY at this final step, for visualization

Per the brief: PySpark for all large-scale work, Pandas/matplotlib/seaborn only for the final plotting step.

In [ ]:
import pandas as pd

viz_df = joined_spark.toPandas()
viz_df.head()

## 11. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fare vs Reliability scatter
sns.scatterplot(data=viz_df, x="avg_fare", y="reliability_score", size="trip_count", ax=axes[0,0], legend=False)
axes[0,0].set_title("Fare vs Reliability Score")
axes[0,0].set_xlabel("Average Fare (GBP)")
axes[0,0].set_ylabel("Reliability Score (lower = more reliable)")

# Trip count by route (top 10)
top10 = viz_df.sort_values("trip_count", ascending=False).head(10)
sns.barplot(data=top10, x="trip_count", y="route", ax=axes[0,1], color="steelblue")
axes[0,1].set_title("Top 10 Routes by Trip Count")

# Disruption count by route (non-zero only)
disrupted = viz_df[viz_df["disruption_count"] > 0].sort_values("disruption_count", ascending=False)
sns.barplot(data=disrupted, x="disruption_count", y="route", ax=axes[1,0], color="indianred")
axes[1,0].set_title("Disruption Count by Route (routes with disruptions only)")

# Distribution of scheduled times (from PySpark-computed stats, plotted as histogram of a sample)
sample_times = tt.select("scheduled_time_sec").sample(fraction=0.05, seed=42).toPandas()
sns.histplot(sample_times["scheduled_time_sec"] / 3600, bins=30, ax=axes[1,1], color="seagreen")
axes[1,1].set_title("Distribution of Scheduled Stop Times (5% sample)")
axes[1,1].set_xlabel("Hour of Day")

plt.tight_layout()
plt.savefig(r'D:\Dataset\output\eda_visualizations.png', dpi=150)
plt.show()

## 12. Save final EDA table for the report

In [ ]:
import os
os.makedirs(r'D:\Dataset\output', exist_ok=True)
viz_df.to_csv(r'D:\Dataset\output\eda_route_summary.csv', index=False)
print("Saved eda_route_summary.csv and eda_visualizations.png to D:\\Dataset\\output")

## Next steps

- Take a Spark UI screenshot (http://localhost:4040) while Section 8's join cell is running -- this notebook is a good place to do it, since it has the heaviest Spark workload
- Use `eda_visualizations.png` and the statistics from Sections 2-6 in the report's Exploratory Data Analysis and Results sections
